# Stage 6 — Model Development: Gradient Boosting (Member 4)
## Online Shopper Purchase Likelihood Prediction System
**IT3051 – Fundamentals of Data Mining | Mini Project 2026 | Group DS_WE_01.02**

This notebook covers Stage 6 for my assigned algorithm, **Gradient Boosting**, following the group's shared
setup: the same engineered training data, 5-fold stratified cross-validation, preprocessing fit inside each
fold, and purchase-class F1 as the main metric (with precision, recall, ROC-AUC and average precision also
reported).

**The test set is not touched anywhere in this notebook.** It stays untouched until the group selects a final
model together.


## Setup

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline

from src.preprocessing.encoding_scaling import CATEGORICAL_COLUMNS, build_preprocessor

RANDOM_STATE = 42


## Load the shared training data
Using the engineered data (before encoding/scaling), since the preprocessor must be fit inside each
cross-validation fold, not once on the whole training set beforehand.

In [2]:
X_train = pd.read_csv("../data/processed/X_train_engineered.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze("columns").astype(int)

# Row alignment check — the two files must line up row-for-row before anything else happens.
assert len(X_train) == len(y_train), "X_train and y_train have different numbers of rows."
assert "Revenue" not in X_train.columns, "Revenue must not be in the input features."

numeric_columns = [c for c in X_train.columns if c not in CATEGORICAL_COLUMNS]

print("Training rows:", len(X_train))
print("Purchase rate: {:.3f}".format(y_train.mean()))
print("Numeric features:", len(numeric_columns), "| Categorical features:", len(CATEGORICAL_COLUMNS))


Training rows: 9764
Purchase rate: 0.156
Numeric features: 13 | Categorical features: 8


## Shared cross-validation setup
5-fold stratified cross-validation, shuffled, with a fixed random seed — identical for every member's model so
results are genuinely comparable.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

SCORING = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}


def summarise(scores):
    """Turn cross_validate's raw fold scores into mean +/- std for each metric."""
    row = {}
    for metric in SCORING:
        values = scores["test_" + metric]
        row[metric + "_mean"] = values.mean()
        row[metric + "_std"] = values.std()
    return row


## Shared baseline — "always predict no purchase"
Run once here (and reusable by the rest of the group) to show concretely why accuracy alone would be
misleading: this baseline gets ~84.5% accuracy while being completely useless, since it never finds a single
real purchaser.

In [4]:
dummy_pipeline = Pipeline([
    ("prep", build_preprocessor(numeric_columns)),
    ("model", DummyClassifier(strategy="most_frequent")),
])

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    dummy_scores = cross_validate(dummy_pipeline, X_train, y_train, cv=cv, scoring=SCORING)
dummy_result = summarise(dummy_scores)

print("Always-predict-no-purchase baseline (5-fold CV):")
for k, v in dummy_result.items():
    print(f"  {k}: {v:.4f}")


Always-predict-no-purchase baseline (5-fold CV):
  f1_mean: 0.0000
  f1_std: 0.0000
  precision_mean: 0.0000
  precision_std: 0.0000
  recall_mean: 0.0000
  recall_std: 0.0000
  roc_auc_mean: 0.5000
  roc_auc_std: 0.0000
  pr_auc_mean: 0.1563
  pr_auc_std: 0.0002


A warning normally appears here ("Precision is ill-defined... no predicted samples") because this baseline
never predicts the positive class at all, so precision has nothing to divide by. The warning is expected and is
suppressed above only for readability — it is not a sign of an error in the code.

As expected, this baseline's F1, precision, and recall for the purchase class are all 0 (or NaN, since it
never predicts a positive case at all) — it cannot find any purchasers, no matter how good accuracy looks in
isolation. This is the concrete evidence for why accuracy is not used as a comparison metric anywhere else in
this project.

## Explaining the algorithm — Gradient Boosting
Gradient Boosting builds an ensemble of decision trees **sequentially**, rather than training many independent
trees and voting (as Random Forest does). Each new tree is trained specifically to correct the mistakes
(residual errors) of the trees built before it, with each tree's contribution scaled down by a learning rate
before being added to the overall prediction. This makes it well suited to binary classification problems like
this one, because it can build up a complex, non-linear decision boundary gradually and tends to perform well
on structured/tabular data such as this dataset, with a mix of numeric and categorical features.

## Untuned Gradient Boosting — default settings

In [5]:
gb_pipeline = Pipeline([
    ("prep", build_preprocessor(numeric_columns)),
    ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
])

gb_scores = cross_validate(gb_pipeline, X_train, y_train, cv=cv, scoring=SCORING)
gb_untuned_result = summarise(gb_scores)

print("Untuned Gradient Boosting (5-fold CV, default settings, no imbalance handling):")
for k, v in gb_untuned_result.items():
    print(f"  {k}: {v:.4f}")


Untuned Gradient Boosting (5-fold CV, default settings, no imbalance handling):
  f1_mean: 0.6520
  f1_std: 0.0222
  precision_mean: 0.7245
  precision_std: 0.0280
  recall_mean: 0.5931
  recall_std: 0.0227
  roc_auc_mean: 0.9295
  roc_auc_std: 0.0064
  pr_auc_mean: 0.7451
  pr_auc_std: 0.0217


In [6]:
results_table = pd.DataFrame([
    {"model": "Baseline (always 'no purchase')", **dummy_result},
    {"model": "Gradient Boosting (untuned)", **gb_untuned_result},
])
results_table.round(4)


,model,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,Baseline (always 'no purchase'),0.000,0.0000,0.0000,0.000,0.0000,0.0000,0.5000,0.0000,0.1563,0.0002
1,Gradient Boosting (untuned),0.652,0.0222,0.7245,0.028,0.5931,0.0227,0.9295,0.0064,0.7451,0.0217


## Observations

**Observation 1 — strengths:** the untuned Gradient Boosting model clears the baseline by a wide margin: F1 of
0.652 versus the baseline's 0.000, precision of 0.725, and recall of 0.593. Its ROC-AUC (0.930) and PR-AUC
(0.745) are both far above the baseline's 0.500 and 0.156, showing it can meaningfully separate purchasers from
non-purchasers even before any tuning or imbalance handling is applied.

**Observation 2 — errors / limitations:** recall (0.593) is noticeably lower than precision (0.725) at the
default decision threshold, meaning the model is currently more cautious — it misses roughly 4 in 10 real
purchasers rather than over-flagging non-buyers. This is the natural starting point for Stage 7, where balanced
sample weighting will be tested to see whether that precision/recall trade-off can be improved without hurting
overall F1.


---
## Summary of this notebook

| Model | Purpose |
|---|---|
| Baseline ("always no purchase") | Shows why accuracy alone is misleading for this dataset |
| Gradient Boosting (untuned) | My Stage 6 starting point — recorded as mean ± std across 5 folds, to be compared against the tuned version in Stage 7 |

This untuned result is the number Stage 7's "tuned vs. untuned" comparison will be measured against, using the
exact same features and imbalance setting in both runs.
